# UAV Object Detection — Faz 3 Baseline (Colab / A100)

TEKNOFEST Havacılıkta Yapay Zeka — Görev 1. **YOLO11s + P2 başlığı**, tam kare fine-tune (SAHI YOK — o Faz 5).

Bu skor sonraki tüm fazların referans noktası.

**Sıra:** Runtime > Change runtime type > A100 GPU. Sonra hücreleri sırayla çalıştır.

## 1. Depo + Drive
Repo private — Drive'a zip'leyip yükle. Veri seti (`data/uav_ldz`, `data/coco`, `data/sanity`) da Drive'da.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# repo zip'ini aç
!mkdir -p /content/proje && unzip -q -o /content/drive/MyDrive/uav-object-detection.zip -d /content/proje
%cd /content/proje
!pip install -q -e .

## 2. Bağımlılıklar + ortam doğrulama

In [ ]:
!pip install -q -r requirements-train.txt
import torch, ultralytics
print('ultralytics', ultralytics.__version__)
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 3. Veri
`data/` klasörünü Drive'dan kopyala. Beklenen: `data/uav_ldz/{train,val,test}/images` + `data/coco/instances_*.json` + `data/sanity/`.

`configs/data.yaml` `path:` satırını Colab yoluna çevir (yerel mutlak yol işe yaramaz).

In [ ]:
!cp -r /content/drive/MyDrive/uav_data/* data/

# data.yaml path'ini Colab'a göre yaz
cfg = '''path: /content/proje/data/uav_ldz
train: train/images
val: val/images
test: test/images
names:
  0: vehicle
  1: human
  2: uap
  3: uai
'''
open('configs/data.yaml', 'w').write(cfg)

!ls data/uav_ldz/*/images | head && echo '---' && ls data/coco

## 4. Eğitim (Faz 3 baseline)
`configs/baseline.yaml`: yolo11s+P2, imgsz 1280, 150 epoch, batch 32, COCO→bizim veri.
A100'de ~1-1.5 saat beklenir. `runs/faz3/baseline-yolo11s-p2/` altına yazar.

In [ ]:
!python scripts/train.py --config configs/baseline.yaml

## 5. Resmi referans skor (bizim eval aracı)
Ultralytics val hızlı kontrol; kesin skor = `predict_to_coco.py` → `evaluate.py` (mAP@0.5, sınıf-başı AP, faster-coco-eval çapraz).

In [ ]:
W = 'runs/faz3/baseline-yolo11s-p2/weights/best.pt'

!python scripts/predict_to_coco.py $W test   --out preds_test.json
!python scripts/evaluate.py preds_test.json data/coco/instances_test.json --cross-check --json score_test.json

print('\n===== SANITY (örnek oturum dağılımı) =====')
!python scripts/predict_to_coco.py $W sanity --out preds_sanity.json
!python scripts/evaluate.py preds_sanity.json data/coco/instances_sanity.json --cross-check --json score_sanity.json

## 6. Çıktıları Drive'a kaydet
`best.pt` (repoya girmez — gitignore) + skor JSON'ları + eğitim grafikleri.

In [ ]:
!mkdir -p /content/drive/MyDrive/uav_runs/faz3
!cp -r runs/faz3/baseline-yolo11s-p2 /content/drive/MyDrive/uav_runs/faz3/
!cp score_test.json score_sanity.json /content/drive/MyDrive/uav_runs/faz3/